# COCO Dataset EDA
Research-grade exploratory data analysis for COCO 2017 annotations.

In [ ]:
!pip -q install pycocotools seaborn opencv-python

import os, json, random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pycocotools.coco import COCO
from matplotlib.patches import Rectangle
from PIL import Image

plt.rcParams['figure.figsize']=(10,6)
sns.set_style("whitegrid")


## Download COCO (run once)

In [ ]:
# Uncomment if needed
# !wget http://images.cocodataset.org/annotations/annotations_trainval2017.zip
# !unzip annotations_trainval2017.zip
# !wget http://images.cocodataset.org/zips/train2017.zip
# !unzip train2017.zip


In [ ]:
ann_file='annotations/instances_train2017.json'
img_dir='train2017'

coco=COCO(ann_file)

cats=coco.loadCats(coco.getCatIds())
cat_names=[c['name'] for c in cats]

print("Images:",len(coco.imgs))
print("Annotations:",len(coco.anns))
print("Categories:",len(cat_names))


In [ ]:
records=[]

for ann in coco.anns.values():
    img=coco.imgs[ann['image_id']]
    cat=coco.cats[ann['category_id']]['name']
    x,y,w,h=ann['bbox']
    records.append({
        'image_id':ann['image_id'],
        'category':cat,
        'width':w,
        'height':h,
        'area':ann['area'],
        'img_width':img['width'],
        'img_height':img['height']
    })

df=pd.DataFrame(records)
df.head()


## Dataset Statistics

In [ ]:
display(df.describe())
display(df.isnull().sum())


## Category Distribution

In [ ]:
top=df['category'].value_counts().head(20)
plt.figure(figsize=(12,6))
sns.barplot(x=top.values,y=top.index)
plt.title("Top 20 Categories")
plt.show()

## Bounding Box Area

In [ ]:
plt.figure(figsize=(8,5))
sns.histplot(df['area'],bins=50)
plt.title("Bounding Box Area")
plt.show()

## Bounding Box Width

In [ ]:
sns.histplot(df['width'],bins=50)
plt.title("Bounding Box Width")
plt.show()

## Bounding Box Height

In [ ]:
sns.histplot(df['height'],bins=50)
plt.title("Bounding Box Height")
plt.show()

## Objects per Image

In [ ]:
cnt=df.groupby('image_id').size()
sns.histplot(cnt,bins=40)
plt.title("Objects per Image")
plt.show()

## Image Resolution

In [ ]:
res=pd.DataFrame({'width':[i['width'] for i in coco.imgs.values()],
'height':[i['height'] for i in coco.imgs.values()]})
sns.scatterplot(data=res,x='width',y='height',s=10)
plt.title("Image Resolution Distribution")
plt.show()

## Sample Image with Bounding Boxes

In [ ]:
img_id=random.choice(list(coco.imgs.keys()))
img_info=coco.loadImgs(img_id)[0]
img_path=os.path.join(img_dir,img_info['file_name'])

img=np.array(Image.open(img_path))
anns=coco.loadAnns(coco.getAnnIds(imgIds=img_id))

fig,ax=plt.subplots(figsize=(10,8))
ax.imshow(img)

for ann in anns:
    x,y,w,h=ann['bbox']
    ax.add_patch(Rectangle((x,y),w,h,fill=False,edgecolor='red',linewidth=2))
    ax.text(x,y,coco.cats[ann['category_id']]['name'],color='yellow',fontsize=8)

plt.axis('off')
plt.show()


## EDA Summary

- Dataset overview completed
- Category distribution analyzed
- Bounding-box statistics generated
- Resolution analysis completed
- Objects-per-image distribution analyzed
- Sample annotations visualized

Ready for preprocessing and model training.
